# Map water in Earth Engine with a trained U-Net

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/01_quickstart.ipynb)

This notebook runs a trained water-mapping U-Net directly inside Google Earth Engine on a single 224 x 224 Sentinel-1 chip (10 m pixels). Earth Engine evaluates every layer itself from public weight tables, so you do not need a local model file or a serving endpoint.

Enter your [registered Earth Engine](https://developers.google.com/earth-engine/guides/access) Cloud project ID in step 3, then choose **Runtime -> Run all**. The example export uses about 65 EECU-seconds (64.9 recorded), which is free on noncommercial projects and uses your plan's quota on commercial ones.

![Sun Moon Lake Sentinel-1 radar input, water probability, and water mask computed inside Earth Engine](https://raw.githubusercontent.com/noelEOS/tensor-ee/main/docs/images/water_chip_224.png)
*Example output at Sun Moon Lake, Taiwan (3 January 2024). Prediction, not ground truth. Contains modified Copernicus Sentinel data.*

In [ ]:
# @title 1. Install (about a minute)
!git clone --quiet --depth 1 https://github.com/noelEOS/tensor-ee.git
!pip install --quiet ./tensor-ee folium

import sys
from pathlib import Path

REPOSITORY = next(folder.resolve() for folder in (Path("tensor-ee"), Path("."), Path(".."))
                  if (folder / "demos" / "code_editor" / "segwater.js").is_file())
sys.path.insert(0, str(REPOSITORY))
print("TensorEE is ready.")

In [ ]:
# @title 2. Helpers and plotting
# @markdown Self-contained helpers for waiting on exports and plotting the radar chip and water map.
import time
import numpy as np

TASKS_URL = "https://code.earthengine.google.com/tasks"


class Stop(Exception):
    """Stops a cell with a single readable line instead of a traceback."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


def wait_for(task, poll=20):
    """Wait for an Earth Engine export, printing a status line every minute."""
    started, polls = time.time(), 0
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        time.sleep(poll)
        polls += 1
        if polls % 3 == 0:
            minutes = (time.time() - started) / 60
            print(f"  still running, {minutes:.0f} min ({state.lower()})...", flush=True)
    if state != "COMPLETED":
        raise Stop(f"the export ended {state}: {task.status().get('error_message', '')} (see {TASKS_URL})")


def plot_chip_preview(vv_db, scene_id, grid):
    """Show a quick Sentinel-1 VV preview of the 224 x 224 chip before running the export."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return
    short_scene = scene_id.rsplit("/", 1)[-1]
    fig, ax = plt.subplots(figsize=(4.4, 4.4), layout="constrained")
    im = ax.imshow(vv_db, cmap="gray", vmin=-25, vmax=0, interpolation="nearest", origin="upper")
    ax.set_title(f"Sentinel-1 VV input ({grid.width} x {grid.height} px, {grid.crs})", fontsize=10.5, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
    cb.set_label("Backscatter (dB)")
    fig.supxlabel(short_scene, fontsize=8.5, color="#475569")
    plt.show()


def plot_water_panels(vv_db, probability, mask, scene_id, grid):
    """Plot Sentinel-1 VV, water probability, and binary water mask side by side."""
    try:
        import matplotlib.pyplot as plt
        from matplotlib.colors import LinearSegmentedColormap, ListedColormap
    except ImportError:
        return
    blue = LinearSegmentedColormap.from_list("water", ["#f5f5f5", "#90caf9", "#1565c0"])
    fig, axes = plt.subplots(1, 3, figsize=(11.2, 4.1), layout="constrained")
    fig.suptitle("Trained U-Net inference inside Google Earth Engine", fontsize=13, fontweight="bold")
    panels = [
        (vv_db, "Sentinel-1 VV", "gray", -25, 0, "Backscatter (dB)"),
        (probability, "Water probability", blue, 0, 1, "Class-1 probability"),
        (mask, "Water mask", ListedColormap(["#eeeeee", "#1565c0"]), 0, 1, "0 = non-water   1 = water"),
    ]
    for ax, (values, title, cmap, lo, hi, label) in zip(axes, panels):
        shown = ax.imshow(values, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest", origin="upper")
        ax.set_title(title, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(shown, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
        cb.set_label(label, fontsize=9.5)
        if title == "Water mask":
            cb.set_ticks([0, 1])
    short_scene = scene_id.rsplit("/", 1)[-1]
    fig.supxlabel(f"{grid.crs} | {grid.width} x {grid.height} px at 10 m | {short_scene} | prediction, not ground truth",
                  fontsize=9, color="#475569")
    plt.show()


print("Helpers loaded.")

In [ ]:
# @title 3. Project and location
# @markdown Enter your Cloud project ID. The default point is Sun Moon Lake, Taiwan.
PROJECT = ""  # @param {type:"string"}
LONGITUDE = 120.915  # @param {type:"number"}
LATITUDE = 23.862  # @param {type:"number"}
START_DATE = "2024-01-01"  # @param {type:"date"}
END_DATE = "2024-01-08"  # @param {type:"date"}

ASSET_NAME = "tensor_ee_quickstart_" + time.strftime("%Y%m%d_%H%M%S")

if not PROJECT.strip():
    raise Stop("enter your Cloud project ID in PROJECT above, then run all cells again.")
print(f"Using project {PROJECT} at ({LONGITUDE}, {LATITUDE}), dates {START_DATE} to {END_DATE}.")

In [ ]:
# @title 4. Sign in
import ee

ee.Authenticate()
ee.Initialize(project=PROJECT)
print(f"Signed in to {PROJECT}.")

In [ ]:
# @title 5. Pick a Sentinel-1 chip and preview it
from tensor_ee import Grid

CHIP, PIXEL = 224, 10
zone = min(60, max(1, int((LONGITUDE + 180) // 6) + 1))
crs = f"EPSG:{326 if LATITUDE >= 0 else 327}{zone:02d}"
x, y = ee.Geometry.Point([LONGITUDE, LATITUDE]).transform(crs, 0.01).coordinates().getInfo()
half = CHIP * PIXEL // 2
place = {"crs": crs, "x": round((x - half) / PIXEL) * PIXEL, "y": round((y + half) / PIXEL) * PIXEL}
grid = Grid(crs, (PIXEL, 0, place["x"], 0, -PIXEL, place["y"]), CHIP, CHIP)

scenes = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(grid.geometry()).filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .filter(ee.Filter.eq("resolution_meters", 10))
          .sort("system:time_start").limit(10))


def _coverage(image):
    valid = image.select(["VV", "VH"]).mask().reduce(ee.Reducer.min()).gt(0).unmask(0, False)
    share = valid.rename("valid").reduceRegion(
        reducer=ee.Reducer.mean(), geometry=grid.geometry(), crs=grid.crs,
        crsTransform=list(grid.transform), maxPixels=CHIP * CHIP, tileScale=2).get("valid")
    return ee.Feature(None, {"id": ee.String("COPERNICUS/S1_GRD/").cat(image.get("system:index")), "share": share})


found = scenes.map(_coverage).reduceColumns(ee.Reducer.toList(2), ["id", "share"]).get("list").getInfo()
complete = [scene for scene, share in found if share >= 0.999]
if not complete:
    raise Stop(f"no Sentinel-1 scene covers the whole chip between {START_DATE} and {END_DATE}; widen the dates or move the point.")
scene_id = complete[0]
print(f"Selected {scene_id} ({CHIP} x {CHIP} px in {crs}).")

_names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
vv_preview = ee.data.computePixels({
    "expression": ee.Image(scene_id).select(["VV"]).toFloat(),
    "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": grid.width, "height": grid.height},
             "affineTransform": dict(zip(_names, grid.transform)), "crsCode": grid.crs},
})["VV"]
plot_chip_preview(vv_preview, scene_id, grid)

In [ ]:
# @title 6. Build the U-Net graph (nothing runs yet)
import ee
from demos.code_editor import build as segwater_module
from tensor_ee import Grid

sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))
TEMPLATE, METADATA = segwater_module.parse_module(
    (REPOSITORY / "demos" / "code_editor" / "segwater.js").read_text())
BANDS = list(segwater_module.OUTPUT_BANDS)

grid = Grid(place["crs"], (10, 0, place["x"], 0, -10, place["y"]), 224, 224)
text = segwater_module.substitute(TEMPLATE, grid.crs, scene_id,
                                  (place["x"], place["y"]), (place["x"], place["y"]))
chip = ee.Image(ee.deserializer.fromJSON(text))
prediction = (ee.ImageCollection.fromImages([chip]).mosaic().select(BANDS)
              .setDefaultProjection(grid.projection()).clip(grid.geometry())
              .updateMask(grid.mask()).toFloat())
print(f"Graph ready ({len(METADATA['weightAssets'])} public weight tables). Step 7 runs the export.")

In [ ]:
# @title 7. Run the model (batch export, ~65 EECU-seconds)
asset_id = f"projects/{PROJECT}/assets/{ASSET_NAME}"
task = ee.batch.Export.image.toAsset(
    image=prediction.set({"source_scene": scene_id}), description="tensor_ee_quickstart",
    assetId=asset_id, region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
    maxPixels=224 * 224 * len(BANDS), pyramidingPolicy={".default": "sample"})
task.start()
print(f"Export started. Track it here or at {TASKS_URL}")
wait_for(task)
print(f"Saved water map to {asset_id}")

In [ ]:
# @title 8. Plot the water map
import folium

if "task" in globals() and task.status()["state"] != "COMPLETED":
    wait_for(task)
elif "task" not in globals():
    try:
        ee.data.getAsset(f"projects/{PROJECT}/assets/{ASSET_NAME}")
    except ee.EEException:
        raise Stop("run step 7 first so the water map asset exists.") from None

asset_id = f"projects/{PROJECT}/assets/{ASSET_NAME}"
result = ee.Image(asset_id)

_names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
pixels = ee.data.computePixels({
    "expression": ee.Image(scene_id).select(["VV"]).addBands(result.select(["water_probability", "water_mask"])).toFloat(),
    "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": grid.width, "height": grid.height},
             "affineTransform": dict(zip(_names, grid.transform)), "crsCode": grid.crs},
})
plot_water_panels(pixels["VV"], pixels["water_probability"], pixels["water_mask"], scene_id, grid)


def _layer(image, vis, name, show=True):
    url = ee.Image(image).getMapId(vis)["tile_fetcher"].url_format
    return folium.raster_layers.TileLayer(
        tiles=url, name=name, overlay=True, show=show,
        attr="Google Earth Engine; contains modified Copernicus Sentinel data")


centre_lon, centre_lat = grid.geometry().centroid(1).transform("EPSG:4326", 1).coordinates().getInfo()
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
water_map = folium.Map(location=[centre_lat, centre_lon], zoom_start=14, tiles=None)
folium.raster_layers.TileLayer(tiles=ESRI.format("Canvas/World_Light_Gray_Base"),
                               attr="Tiles &copy; Esri", name="Esri Light Gray").add_to(water_map)
folium.raster_layers.TileLayer(tiles=ESRI.format("World_Imagery"),
                               attr="Tiles &copy; Esri", name="Esri World Imagery").add_to(water_map)
_layer(ee.Image(scene_id).select("VV"), {"min": -25, "max": 0}, "Sentinel-1 VV (dB)").add_to(water_map)
_layer(result.select("water_probability"), {"min": 0, "max": 1, "palette": ["ffffff", "08306b"]},
       "Water probability").add_to(water_map)
_layer(result.select("water_mask").selfMask(), {"palette": ["00bfff"]}, "Water mask", show=False).add_to(water_map)
folium.LayerControl().add_to(water_map)
water_map

## Notes

- **Agreement vs. accuracy:** At Sun Moon Lake, this Earth Engine graph matched PyTorch within 1.35e-5 in the logits with identical water masks ([validation](https://github.com/noelEOS/tensor-ee/blob/main/docs/validation.md)). Numerical agreement confirms Earth Engine ran the same math; classification accuracy still depends on the trained weights ([model notes](https://github.com/noelEOS/tensor-ee/blob/main/docs/model.md)).
- **Batch export and cleanup:** Running the full network interactively hits Earth Engine's memory cap, so step 7 exports an image asset. Delete the asset from the Code Editor's **Assets** tab when you no longer need it.
- **Next:** check your own architecture in [Notebook 02](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb) or export a trained CNN in [Notebook 03](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb).